# 03_05 — Orquestación del mapa operativo final

Demostración del flujo operativo final encapsulado en una sola función.

## Configuración

In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data_catalog.csv").exists():
            return candidate
    raise FileNotFoundError("No se encontro data_catalog.csv en los padres.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

PosixPath('/Users/hugo/TFM_parking_madrid')

## Ejecución única

In [2]:
from src.pipelines.operational_map import build_operational_ser_emt_realtime_map

result = build_operational_ser_emt_realtime_map(
    root=ROOT,
    scenario_datetime=None,
    write_outputs=True,
)

type(result).__name__

/opt/anaconda3/envs/tfm-parking/lib/python3.11/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


'OperationalSEREMTMapResult'

## Metadata

In [3]:
pd.DataFrame(
    [{"campo": key, "valor": value} for key, value in result.metadata.items()]
)

,campo,valor
0,scenario_datetime_requested,2026-07-08 18:54:56.832301
1,scenario_datetime_used,2026-07-08 18:30:00
2,intervalo_inicio,2026-07-08 18:30:00
3,intervalo_fin,2026-07-08 19:00:00
4,intervalo_ajustado_30min,True
5,emt_query_timestamp_utc,2026-07-08T16:54:59.132704+00:00
6,emt_moment_min,2026-07-08 18:43:46+02:00
7,emt_moment_max,2026-07-08 18:54:28+02:00
8,write_outputs,True
9,write_realtime_snapshots,True


## Checks combinados

In [4]:
result.checks

,component,check_id,status,detail,critical
0,ser_proxy,scenario_in_calendar_2023_2026,OK,year=2026,True
1,ser_proxy,scenario_in_ser_observable_window,OK,ventana=09:00-21:00,True
2,ser_proxy,calendar_weekday_encoding_consistent,OK,calendario=3; m0=2,True
3,ser_proxy,scenario_after_training_end,OK,training_period_end=2026-03-31 20:30:00; scena...,True
4,ser_proxy,m0_returns_expected_barrios,OK,rows=65; barrios=65; expected=65,True
...,...,...,...,...,...
75,orchestration,output_exists_raw_xml,OK,data/raw/emt/emt_aparcamientos_rotacionales_ti...,True
76,orchestration,output_exists_interim_realtime,OK,data/interim/emt/emt_aparcamientos_rotacionale...,True
77,orchestration,output_exists_joined_inventory,OK,data/interim/emt/emt_aparcamientos_rotacionale...,True
78,orchestration,output_exists_html_ser_emt_tiempo_real_proxy,OK,reports/maps/mapa_integrado_ser_proxy_emt_tiem...,True


In [5]:
checks_summary = (
    result.checks
    .groupby(["component", "status"], dropna=False)
    .size()
    .reset_index(name="n_checks")
)
checks_summary

,component,status,n_checks
0,emt_realtime,OK,18
1,emt_realtime,WARNING,2
2,map,OK,31
3,map,WARNING,1
4,orchestration,OK,6
5,ser_proxy,OK,20
6,ser_proxy,WARNING,2


In [6]:
critical_failures = result.checks.loc[
    result.checks["critical"].eq(True)
    & result.checks["status"].eq("FAIL")
].copy()
critical_failures

,component,check_id,status,detail,critical


In [7]:
result.diagnostics["map_layer_coverage"]

,layer,n_rows,available
0,prediction_barrios,65,True
1,emt_inventory,85,True
2,emt_map,74,True
3,emt_realtime_live_all,13,True
4,emt_realtime_map,13,True


## Outputs

In [8]:
result.outputs

,component,output,path,exists,size_mb
0,emt_realtime,raw_xml,data/raw/emt/emt_aparcamientos_rotacionales_ti...,True,0.049
1,emt_realtime,interim_realtime,data/interim/emt/emt_aparcamientos_rotacionale...,True,0.016
2,emt_realtime,joined_inventory,data/interim/emt/emt_aparcamientos_rotacionale...,True,0.041
3,map,html_ser_emt_tiempo_real_proxy,reports/maps/mapa_integrado_ser_proxy_emt_tiem...,True,19.584
4,map,png_emt_tiempo_real_zoom,reports/figures/emt_tiempo_real/mapa_emt_tiemp...,True,3.173


## Lectura metodológica

- SER proxy se calcula para el intervalo SER de 30 minutos correspondiente a la hora solicitada.
- Tiempo real municipal se consulta en vivo y no es predicción.
- La integración es cartográfica, no un modelo conjunto SER+EMT.
- `write_outputs=False` permite ejecutar el flujo sin escribir artefactos.